# RULE-BASED CELL TYPING WITH BANKSY

This notebook provides the workflow from AnnData object to BANKSY domains, rule-based cell typing, annotation of BANKSY domains as B-cell follicles, and spatially informed cell typing.

Your AnnData object should contain marker positivity information) from the compute_positivity_matrix function in annotation.py).

In [ ]:
import sys
from pathlib import Path
import gc
import os
import numpy as np
import scanpy as sc
import pandas as pd
import anndata as ad

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Common folders used throughout this notebook.
# Change if your folder layout is different.
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "anndata"
RESULTS_DIR = PROJECT_ROOT / "results"


# BANKSY
For this you need a saved .h5ad file containing an anndata object, and Python 12 is required (not a later version of Python). Note that BANKSY only identifies spatial domains and not cell types.

First, load your H5AD file and choose a "base name" to easily keep track of your file and plot naming.

In [ ]:
# List of all samples to process, with their raw filename, input suffix,
# and output name.
# To add, remove, or rename a sample, edit SAMPLES in scripts/celltype_config.py.

from scripts.celltype_config import SAMPLES

In [ ]:
# EDIT: pick which sample you're working on for the single-sample steps below.
basename = "IHOPE14_MedLN_TopRight"  # For example


In [ ]:
from banksy.initialize_banksy import initialize_banksy
from banksy.run_banksy import run_banksy_multiparam

# Path to the h5ad file for the sample chosen above.
h5ad_file = DATA_DIR / "zscore_log2" / f"{basename}_filtered_zscore_GMM_IHOPE_celltypes.h5ad"

# Load your data
adata = sc.read_h5ad(str(h5ad_file))

# Initialize BANKSY
coord_keys = ('x', 'y', 'spatial')
banksy_dict = initialize_banksy(
    adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay='scaled_gaussian'
)

# Run BANKSY clustering

results_df = run_banksy_multiparam(
    adata,
    banksy_dict,
    lambda_list=[0.2],
    resolutions=[0.5, 1.0]
)


In [ ]:
from banksy.main import median_dist_to_nearest_neighbour
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from banksy.plot_banksy import plot_results

# Parameters for BANKSY
coord_keys = ('x', 'y', 'spatial')
k_geom = 15
max_m = 1
nbr_weight_decay = "scaled_gaussian"
lambda_list = [0.8]
resolutions = [0.5]       # Leiden clustering resolution
pca_dims = [20]             # Try a lower number?
cluster_algorithm = 'leiden'
cmap = 'tab20'             # color map for spatial plotting
save_path = None           # e.g., "./BANKSY_results" if you want to save figures

# Compute neighbor distances
nbrs = median_dist_to_nearest_neighbour(adata, key=coord_keys[2])

# Initialize BANKSY
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=False,
    plt_nbr_weights=True,
    plt_agf_angles=False,
    plt_theta=False
)

# Generate BANKSY matrix
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata,
    banksy_dict,
    lambda_list,
    max_m
)

# Dimensionality reduction by UMAP
pca_umap(
    banksy_dict,
    pca_dims=pca_dims,
    add_umap=True
)

# Run Leiden clustering
results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions=resolutions,
    num_nn=50,
    num_iterations=-1,
    partition_seed=1234,
    match_labels=True,
    max_labels=None
)

# Map clusters back to adata
cluster_labels = results_df.labels[results_df.index[0]].dense
adata.obs['banksy_domain'] = cluster_labels.astype(str)

# Optional: visualize
sc.pl.spatial(adata, color='banksy_domain', spot_size=30, title='BANKSY Domains')

# Optional: use BANKSY plotting function
if save_path is not None:
    os.makedirs(save_path, exist_ok=True)
    weights_graph = banksy_dict['scaled_gaussian']['weights'][1]
    plot_results(
        results_df[results_df['num_labels']==len(np.unique(cluster_labels))],
        weights_graph,
        cmap,
        match_labels=True,
        coord_keys=coord_keys,
        max_num_labels=max_num_labels,
        save_path=save_path,
        save_fig=True,
        save_fullfig=True,
        dataset_name='Sample',
        save_labels=True
    )

print(f"BANKSY complete for {basename}! Domains added to `adata.obs['banksy_domain']`")


Save anndata object with BANKSY domains

In [ ]:
from scripts.anndata_helpers import save_h5ad

h5adpath = DATA_DIR / "zscore_log2" / f"{basename}_filtered_zscore_banksy.h5ad"
save_h5ad(adata, str(h5adpath))


# Rule-based cell typing

Each cell type will correspond to a column in the AnnData object, with a boolean True/False for every cell type in every cell.

On the lineage ("type") level, the labels are exclusive.

On the intermediate and subtype levels, labels may overlap.

Set file name/path and basename if needed:

In [ ]:
# Load AnnData
# EDIT: change basename to the sample you want to load.
# NOTE: this used to be a fixed filename that ignored the basename you set
# below, so changing basename here didn't actually load a different file.
# Fixed so the filename now follows basename, like the equivalent cell
# further down in this notebook.
basename = "IHOPE14_MedLN_BottomRight"

adata = sc.read_h5ad(
    str(DATA_DIR / "zscore_log2" / "celltyped" / f"{basename}_filtered_log2_zscore_banksy_celltypes.h5ad")
)
print(f"Base name: {basename}")


If the AnnData already has cell types, remove them:

In [ ]:
cols_to_drop = [
    c for c in adata.obs.columns
    if c.startswith(("type_", "intermediate_", "subtype_", "state_"))
]

adata.obs.drop(columns=cols_to_drop, inplace=True)

print(f"Removed {len(cols_to_drop)} old annotation columns")

**Cell typing**

In [ ]:
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE

adata = assign_cell_types_bool_IHOPE(adata)

Save:

In [ ]:
from scripts.anndata_helpers import save_h5ad

celltype_path = DATA_DIR / "zscore_log2" / "celltyped" / f"{basename}_zscore_banksy_celltypes.h5ad"

save_h5ad(adata, str(celltype_path))


# B cell follicle domain annotation

Annotate BANKSY domains, here as B-cell follicles. This analysis is based on B cell enrichment across domains, you can then manually select the domains that are likely to be B cell follicles. You can also choose to set a B cell percentage threshold for what is annotated as a follicle.

In [ ]:
# Load AnnData if needed

# EDIT: change basename to the sample you want to load.
basename = "IHOPE39_MesLN_B"

adata = sc.read_h5ad(str(DATA_DIR / "zscore_log2" / "celltyped" / f"{basename}_filtered_log2_zscore_banksy_celltypes.h5ad"))


Rank domains by B cell fraction:

In [ ]:
from scripts.banksy_domains import compute_domain_bcell_stats, plot_domains_by_bcell_fraction

stats_df = compute_domain_bcell_stats(adata)
plot_domains_by_bcell_fraction(adata, stats_df, cmap="coolwarm")

Manually select top domains and add to anndata

In [ ]:
# EDIT: BANKSY domain IDs (as strings) to keep as follicle domains for this sample.
top_domains = ['3']  # <-- you set this per sample, within ''


In [ ]:
from scripts.banksy_domains import assign_bcell_follicles

adata = assign_bcell_follicles(
    adata,
    follicle_domains=top_domains,
    banksy_domain_key="banksy_domain",
    output_key="B_follicle",
)

Quick visual check of the follicle domain:

In [ ]:
from scripts.banksy_domains import plot_domain_mask
plot_domain_mask(adata, top_domains)

Plot of the B cells within the selected domain


In [ ]:
from scripts.banksy_domains import plot_bcell_follicles

plot_bcell_follicles(adata, sample_name = basename)

Save:

In [ ]:
from scripts.anndata_helpers import save_h5ad

follicle_path = DATA_DIR / "zscore_log2" / "celltyped" / f"{basename}_celltypes_follicledomains.h5ad"

save_h5ad(adata, str(follicle_path))


# Spatially informed cell typing
Finally, it's time to add the spatially defined cell types.

First plot the cell types that are to be spatially filtered:

In [ ]:
from scripts.banksy_domains import plot_B_subtypes_over_follicle

b_subtype_cols = {
    "Naive": "subtype_B_naive",
    "GC": "subtype_B_GC",
    "Plasmablast": "subtype_B_Plasmablast",
}

# Before spatial restriction, columns hold the marker-level calls
plot_B_subtypes_over_follicle(
    adata,
    subtype_cols=b_subtype_cols,
    follicle_key="B_follicle",
    sample_name=basename,
    title_suffix="before",
    size=0.3,
)

**TfH-like** restricted to inside follicles

In [ ]:
from scripts.celltype_rules_IHOPE import add_TfH_like_cells

adata = add_TfH_like_cells(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

GC and Plasmablast B cells, refined by follicle location

In [ ]:
#GC-B AND PLASMABLASTS
from scripts.celltype_rules_IHOPE import add_spatial_B_context

adata = add_spatial_B_context(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

Visualize after filtering:

In [ ]:
# After spatial restriction, the same columns hold the follicle-refined calls
plot_B_subtypes_over_follicle(
    adata,
    subtype_cols=b_subtype_cols,
    follicle_key="B_follicle",
    sample_name=basename,
    title_suffix="after",
    size=0.3,
)

Save cell type information:

In [ ]:
final_path = DATA_DIR / f"{basename}_follicledomains_spatialcelltypes.h5ad"

save_h5ad(adata, str(final_path))


# Summarize

Generate a CSV file with cell type counts and percentages on all levels. This can be used in later analysis of the entire dataset.

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_zscore_summary.csv"
)

**Generate a summary of BANKSY domains**

This step applies a loop to all processed samples. Run after applying BANKSY and cell typing to all samples.

In [ ]:
ADATA_DIR = DATA_DIR / "zscore_log2" / "celltyped"

# Make sure they all follow the same naming convention and are not duplicated, then:
files = sorted(ADATA_DIR.glob("*_celltypes_follicledomains.h5ad"))

print(f"Found {len(files)} files:\n")

for f in files:
    print(f.name)

summary = []

for file in files:
    print(f"Processing {file.name}")

    adata = sc.read_h5ad(file)

    basename = file.stem.replace(
        "_celltypes_follicledomains", ""
    )

    total_domains = adata.obs["banksy_domain"].nunique()

    follicle_domains = len(
        adata.uns["B_follicle_domains"]
    )

    summary.append({
        "sample": basename,
        "total_banksy_domains": total_domains,
        "selected_follicle_domains": follicle_domains
    })

    del adata
    gc.collect()


Save summary in CSV format:

In [ ]:
summary_df = pd.DataFrame(summary)
summary_df.to_csv(
    RESULTS_DIR / "banksy_domain_summary_test.csv",
    index=False
)
summary_df


**Or loop through all, if you didn't save individual summaries**

In [ ]:
# Change path(s) if needed:
anndata_dir = PROJECT_ROOT / "data" / "processed" / "anndata" / "zscore_log2" / "celltyped"
suffix = "_celltypes_follicledomains.h5ad"

reports_dir = RESULTS_DIR / "reports" / "zscore_log2"

h5ad_files = sorted(anndata_dir.glob(f"*{suffix}"))
print(f"Found {len(h5ad_files)} files to summarize")

failed = []

for i, path in enumerate(h5ad_files, start=1):
    adata = None
    # strip the pipeline suffix so summary filenames read as the sample name
    basename = path.name.replace(suffix, "")
    print(f"[{i}/{len(h5ad_files)}] {basename}")

    try:
        adata = ad.read_h5ad(path)
        summarize_celltypes_IHOPE(adata, filename=basename, output_dir=reports_dir)
    except Exception as e:
        print(f"  failed, {e}")
        failed.append((basename, str(e)))
    finally:
        del adata
        gc.collect()

print("\nDone")
if failed:
    print(f"{len(failed)} files failed")
    for name, err in failed:
        print(f"  {name}, {err}")
else:
    print("All files summarized")
